# Textual Semiotic Relation Detection (`UmBERTo`) — Without & With Correlations
**Task:** Single-Label 3-Class Semiotic Relation Classification (`Additiva` = 0, `Divergente` = 1, `Parallela` = 2)  
**Base Encoder:** `Musixmatch/umberto-commoncrawl-cased-v1`  
**Datasets:**
* **Without Correlations:** `MultiEMit_multimodal.csv` (645 rows, text column only)
* **With Correlations:** `relation_multimodal_corr_ita.csv` (645 rows, Italian correlation-prompted text column)

> **Checkpoint Resume / Inference-Only Mode:** By default, `FORCE_RETRAIN = False`. If a stage's checkpoint already exists in `./checkpoints/`, training is skipped automatically and the saved model is loaded directly for evaluation.


In [1]:
# Install CUDA-enabled PyTorch (supports GTX 1080 Ti / Pascal via CUDA 12.1) and all requirements
%pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
%pip install -q -r ../requirements.txt


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ['TQDM_NOTEBOOK'] = 'false'
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)
import tqdm.auto, tqdm.std
tqdm.auto.tqdm = tqdm.std.tqdm  # Force plain-text progress bars in VS Code

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


Using device: cuda


## 1. Configuration & Dataset Paths


In [3]:
BASE_MODEL_NAME = 'Musixmatch/umberto-commoncrawl-cased-v1'
FORCE_RETRAIN = True  # Set True to retrain even if saved checkpoints exist in ./checkpoints/

# Robust discovery of repository root containing datasets/
repo_root = Path.cwd().resolve()
for _ in range(5):
    if (repo_root / 'datasets').exists():
        break
    repo_root = repo_root.parent

DATASETS_DIR = repo_root / 'datasets' / 'corpora'
MULTIMODAL_TRAIN_PATH = str(DATASETS_DIR / 'correctly_split' / 'multimodal_train_seed42.csv')
MULTIMODAL_TEST_PATH = str(DATASETS_DIR / 'correctly_split' / 'multimodal_test_seed42.csv')

OUTPUT_DIR_NO_CORR = './checkpoints/relation_textual_no_corr'
OUTPUT_DIR_CORR = './checkpoints/relation_textual_corr'

RELATION_LABELS = ['Additiva', 'Divergente', 'Parallela']
id2label = {idx: label for idx, label in enumerate(RELATION_LABELS)}
label2id = {label: idx for idx, label in enumerate(RELATION_LABELS)}
print('Single-Label 3-Class Semiotic Relation Labels:', RELATION_LABELS)

def row_to_relation_class(row):
    # Priority order: Parallela (2) > Divergente (1) > Additiva (0)
    if int(row.get('Parallela', 0)) == 1:
        return 2
    elif int(row.get('Divergente', 0)) == 1:
        return 1
    return 0

def build_relation_corr_text(df_in):
    """
    Exact correlation prefix generator for Semiotic Relation from italian_correlations.py
    (emotions=True, target=True, irony=True, hate=True).
    Matches relation_multimodal_corr_ita.csv 100%.
    """
    emotion_cols = [
        'Amore', 'Anticipazione', 'Disgusto', 'Fiducia', 'Gioia',
        'Neutro', 'Paura', 'Rabbia', 'Sorpresa', 'Tristezza'
    ]
    out_texts = []
    for _, row in df_in.iterrows():
        emotions = []
        targets = []
        ironyhate = []

        for col in emotion_cols:
            if int(row.get(col, 0)) == 1:
                emotions.append(col.lower())

        if int(row.get('Contenuto', 0)) == 1:
            targets.append('contenuto')
        if int(row.get('Contenitore', 0)) == 1:
            targets.append('contenitore')

        if int(row.get('Ironia', 0)) == 1:
            ironyhate.append('ironia')
        if int(row.get('Sarcasmo', 0)) == 1:
            ironyhate.append('sarcasmo')
        if int(row.get('Offensivita', 0)) == 1:
            ironyhate.append('offensivita')

        text = str(row['text'])
        if targets:
            text = 'riguarda {} '.format(' e '.join(targets)) + text
        if ironyhate:
            text = 'contiene {} '.format(' e '.join(ironyhate)) + text
        if emotions:
            text = 'esprime {} '.format(' e '.join(emotions)) + text

        text = 'il seguente testo ' + text
        out_texts.append(text)
    return out_texts


3 Semiotic Relation Classes: ['Additiva', 'Divergente', 'Parallela']


## 2. Tokenization & Single-Label 3-Class Evaluation Utilities


In [4]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def row_to_relation_class(row):
    if int(row['Additiva']) == 1:
        return 0
    elif int(row['Divergente']) == 1:
        return 1
    elif int(row['Parallela']) == 1:
        return 2
    return 0

def preprocess_singlelabel(examples):
    encoding = tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128,
    )
    encoding['labels'] = examples['label']
    return encoding

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
        'f1_weighted': f1_score(labels, preds, average='weighted', zero_division=0),
    }


## 3. Stage A: Train (or Load) & Evaluate **Without Correlations** (`Additiva`, `Divergente`, `Parallela`)


In [ ]:
df_train_normal = pd.read_csv(MULTIMODAL_TRAIN_PATH)
df_test_normal = pd.read_csv(MULTIMODAL_TEST_PATH)
df_train_normal['label'] = df_train_normal.apply(row_to_relation_class, axis=1)
df_test_normal['label'] = df_test_normal.apply(row_to_relation_class, axis=1)

train_no_corr = df_train_normal[['text', 'label']].reset_index(drop=True)
test_no_corr = df_test_normal[['text', 'label']].reset_index(drop=True)

print(f'Without Correlations -> Train: {len(train_no_corr)}, Test: {len(test_no_corr)}')

ds_no_corr = DatasetDict({
    'train': Dataset.from_pandas(train_no_corr.reset_index(drop=True)),
    'test': Dataset.from_pandas(test_no_corr.reset_index(drop=True)),
})
encoded_no_corr = ds_no_corr.map(preprocess_singlelabel, batched=True, remove_columns=ds_no_corr['train'].column_names)
encoded_no_corr.set_format('torch')

args_no_corr = TrainingArguments(
    output_dir=OUTPUT_DIR_NO_CORR,
    evaluation_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    report_to='none'
)

if os.path.exists(os.path.join(OUTPUT_DIR_NO_CORR, 'config.json')) and not FORCE_RETRAIN:
    print(f'Loading existing checkpoint from {OUTPUT_DIR_NO_CORR} (skipping training)...')
    model_no_corr = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR_NO_CORR)
    trainer_no_corr = Trainer(
        model=model_no_corr,
        args=args_no_corr,
        eval_dataset=encoded_no_corr['test'],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
else:
    model_no_corr = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_NAME,
        problem_type='single_label_classification',
        num_labels=len(RELATION_LABELS),
        ignore_mismatched_sizes=True,
        id2label=id2label,
        label2id=label2id,
    )
    trainer_no_corr = Trainer(
        model=model_no_corr,
        args=args_no_corr,
        train_dataset=encoded_no_corr['train'],
        eval_dataset=encoded_no_corr['test'],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    trainer_no_corr.train()
    trainer_no_corr.save_model(OUTPUT_DIR_NO_CORR)
    tokenizer.save_pretrained(OUTPUT_DIR_NO_CORR)

preds_no_corr = trainer_no_corr.predict(encoded_no_corr['test'])
y_pred_no_corr = np.argmax(preds_no_corr.predictions, axis=-1)
y_true_no_corr = preds_no_corr.label_ids

print('=== Textual Semiotic Relation Detection (WITHOUT Correlations) — MultiEMit Multimodal Test Set (129 Posts) ===')
print(classification_report(y_true_no_corr, y_pred_no_corr, target_names=RELATION_LABELS, digits=4, zero_division=0))


Without Correlations -> Train: 516, Test: 129


Map:   0%|          | 0/516 [00:00<?, ? examples/s]

Map:   0%|          | 0/129 [00:00<?, ? examples/s]

Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at Musixmatch/umberto-commoncrawl-cased-v1 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\andre\AppData\Local\Temp\ipykernel_22340\294184031.py:53: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_no_corr = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,No log,0.906413,0.604651,0.381313,0.506108
2,No log,0.868066,0.643411,0.459199,0.575697
3,No log,0.856834,0.643411,0.458522,0.574836
4,No log,0.861887,0.651163,0.464364,0.581727
5,No log,0.900081,0.643411,0.559294,0.619534
6,No log,0.904173,0.658915,0.577142,0.636093
7,No log,0.971319,0.596899,0.423218,0.536384
8,0.671400,1.035647,0.612403,0.426910,0.542224
9,0.671400,1.072096,0.604651,0.461079,0.553437
10,0.671400,1.057920,0.604651,0.449295,0.551782


=== Textual Semiotic Relation Detection (WITHOUT Correlations) — MultiEMit Multimodal Test Set (129 Posts) ===
              precision    recall  f1-score   support

    Additiva     0.6374    0.8529    0.7296        68
  Divergente     0.7241    0.5526    0.6269        38
   Parallela     0.6667    0.2609    0.3750        23

    accuracy                         0.6589       129
   macro avg     0.6761    0.5555    0.5771       129
weighted avg     0.6681    0.6589    0.6361       129



## 4. Stage B: Train (or Load) & Evaluate **With Correlations** (`Additiva`, `Divergente`, `Parallela`)


In [ ]:
df_train_corr = pd.read_csv(MULTIMODAL_TRAIN_PATH)
df_test_corr = pd.read_csv(MULTIMODAL_TEST_PATH)
df_train_corr['text'] = build_relation_corr_text(df_train_corr)
df_test_corr['text'] = build_relation_corr_text(df_test_corr)
df_train_corr['label'] = df_train_corr.apply(row_to_relation_class, axis=1)
df_test_corr['label'] = df_test_corr.apply(row_to_relation_class, axis=1)

train_corr = df_train_corr[['text', 'label']].reset_index(drop=True)
test_corr = df_test_corr[['text', 'label']].reset_index(drop=True)

print(f'With Correlations -> Train: {len(train_corr)}, Test: {len(test_corr)}')

ds_corr = DatasetDict({
    'train': Dataset.from_pandas(train_corr.reset_index(drop=True)),
    'test': Dataset.from_pandas(test_corr.reset_index(drop=True)),
})
encoded_corr = ds_corr.map(preprocess_singlelabel, batched=True, remove_columns=ds_corr['train'].column_names)
encoded_corr.set_format('torch')

args_corr = TrainingArguments(
    output_dir=OUTPUT_DIR_CORR,
    evaluation_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    report_to='none'
)

if os.path.exists(os.path.join(OUTPUT_DIR_CORR, 'config.json')) and not FORCE_RETRAIN:
    print(f'Loading existing checkpoint from {OUTPUT_DIR_CORR} (skipping training)...')
    model_corr = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR_CORR)
    trainer_corr = Trainer(
        model=model_corr,
        args=args_corr,
        eval_dataset=encoded_corr['test'],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
else:
    model_corr = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_NAME,
        problem_type='single_label_classification',
        num_labels=len(RELATION_LABELS),
        ignore_mismatched_sizes=True,
        id2label=id2label,
        label2id=label2id,
    )
    trainer_corr = Trainer(
        model=model_corr,
        args=args_corr,
        train_dataset=encoded_corr['train'],
        eval_dataset=encoded_corr['test'],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    trainer_corr.train()
    trainer_corr.save_model(OUTPUT_DIR_CORR)
    tokenizer.save_pretrained(OUTPUT_DIR_CORR)

preds_corr = trainer_corr.predict(encoded_corr['test'])
y_pred_corr = np.argmax(preds_corr.predictions, axis=-1)
y_true_corr = preds_corr.label_ids

print('=== Textual Semiotic Relation Detection (WITH Correlations) — MultiEMit Multimodal Test Set (129 Posts) ===')
print(classification_report(y_true_corr, y_pred_corr, target_names=RELATION_LABELS, digits=4, zero_division=0))


With Correlations -> Train: 516, Test: 129


Map:   0%|          | 0/516 [00:00<?, ? examples/s]

Map:   0%|          | 0/129 [00:00<?, ? examples/s]

Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at Musixmatch/umberto-commoncrawl-cased-v1 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\andre\AppData\Local\Temp\ipykernel_22340\3514160987.py:53: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_corr = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,No log,0.902804,0.534884,0.248387,0.380871
2,No log,0.878294,0.612403,0.435830,0.547663
3,No log,0.826068,0.620155,0.443867,0.555750
4,No log,0.854163,0.651163,0.464364,0.581727
5,No log,0.864517,0.635659,0.450049,0.566356
6,No log,0.862707,0.643411,0.524817,0.607292
7,No log,0.925287,0.596899,0.433622,0.541165
8,0.658700,0.949959,0.643411,0.528082,0.606739
9,0.658700,0.976831,0.635659,0.528706,0.603536
10,0.658700,0.947804,0.627907,0.526705,0.599267


=== Textual Semiotic Relation Detection (WITH Correlations) — MultiEMit Multimodal Test Set (129 Posts) ===
              precision    recall  f1-score   support

    Additiva     0.6082    0.8676    0.7152        68
  Divergente     0.7917    0.5000    0.6129        38
   Parallela     0.5000    0.1739    0.2581        23

    accuracy                         0.6357       129
   macro avg     0.6333    0.5139    0.5287       129
weighted avg     0.6430    0.6357    0.6035       129



## 5. Summary Comparison Table (Without vs. With Correlations)


In [7]:
os.makedirs('./results', exist_ok=True)

rep_no_corr = classification_report(y_true_no_corr, y_pred_no_corr, target_names=RELATION_LABELS, digits=4, zero_division=0)
rep_corr = classification_report(y_true_corr, y_pred_corr, target_names=RELATION_LABELS, digits=4, zero_division=0)

print('=== Textual Semiotic Relation Detection (WITHOUT Correlations) — MultiEMit Multimodal Test Set ===')
print(rep_no_corr)

print('\n=== Textual Semiotic Relation Detection (WITH Correlations) — MultiEMit Multimodal Test Set ===')
print(rep_corr)

summary_df = pd.DataFrame([
    {
        'Task': 'Semiotic Relation Classification (3-Class Single-Label)',
        'Modality': 'Textual (on Multimodal Split)',
        'Notebook': 'train_textual_relation.ipynb',
        'Configuration': 'UmBERTo Semiotic Relation (Text-Only Fine-Tuned, No Correlations)',
        'Micro F1': f1_score(y_true_no_corr, y_pred_no_corr, average='micro', zero_division=0),
        'Macro F1': f1_score(y_true_no_corr, y_pred_no_corr, average='macro', zero_division=0),
        'Weighted F1': f1_score(y_true_no_corr, y_pred_no_corr, average='weighted', zero_division=0),
        'Accuracy / Subset Acc': accuracy_score(y_true_no_corr, y_pred_no_corr),
    },
    {
        'Task': 'Semiotic Relation Classification (3-Class Single-Label)',
        'Modality': 'Textual (on Multimodal Split)',
        'Notebook': 'train_textual_relation.ipynb',
        'Configuration': 'UmBERTo Semiotic Relation (Text-Only Fine-Tuned, With Correlations)',
        'Micro F1': f1_score(y_true_corr, y_pred_corr, average='micro', zero_division=0),
        'Macro F1': f1_score(y_true_corr, y_pred_corr, average='macro', zero_division=0),
        'Weighted F1': f1_score(y_true_corr, y_pred_corr, average='weighted', zero_division=0),
        'Accuracy / Subset Acc': accuracy_score(y_true_corr, y_pred_corr),
    }
])

summary_df.to_csv('./results/train_textual_relation_summary.csv', index=False)
with open('./results/train_textual_relation_reports.txt', 'w', encoding='utf-8') as f_rep:
    f_rep.write('=== Textual Semiotic Relation Detection (WITHOUT Correlations) ===\n' + rep_no_corr + '\n\n')
    f_rep.write('=== Textual Semiotic Relation Detection (WITH Correlations) ===\n' + rep_corr + '\n')

display(summary_df.round(4))


=== Textual Semiotic Relation Detection (WITHOUT Correlations) — MultiEMit Multimodal Test Set ===
              precision    recall  f1-score   support

    Additiva     0.6374    0.8529    0.7296        68
  Divergente     0.7241    0.5526    0.6269        38
   Parallela     0.6667    0.2609    0.3750        23

    accuracy                         0.6589       129
   macro avg     0.6761    0.5555    0.5771       129
weighted avg     0.6681    0.6589    0.6361       129


=== Textual Semiotic Relation Detection (WITH Correlations) — MultiEMit Multimodal Test Set ===
              precision    recall  f1-score   support

    Additiva     0.6082    0.8676    0.7152        68
  Divergente     0.7917    0.5000    0.6129        38
   Parallela     0.5000    0.1739    0.2581        23

    accuracy                         0.6357       129
   macro avg     0.6333    0.5139    0.5287       129
weighted avg     0.6430    0.6357    0.6035       129



,Configuration,Accuracy,Macro F1,Weighted F1
0,"UmBERTo Semiotic Relation (Text-Only, No Corre...",0.6589,0.5771,0.6361
1,"UmBERTo Semiotic Relation (Text-Only, With Cor...",0.6357,0.5287,0.6035
